In [1]:
import duckdb
import pandas as pd
import os

In [2]:
con = duckdb.connect(database='dados_duckdb.db', read_only=False)

In [16]:
df = con.execute("""
                SELECT *
                FROM (
                    SELECT *, ROW_NUMBER() OVER (PARTITION BY NATBR ORDER BY data_injestao DESC) as row
                    FROM bronze_z0019
                    WHERE data_injestao >= '2025-10-10'
                )
                """).fetchdf()
df.head(10)

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_injestao,row
0,10002,MARTELO,BT50,100,1500,z0019_2.csv,2026-10-10 18:33:25.037856,1
1,10002,MACHADO,BT50,100,100,z0019_1.csv,2026-10-10 18:25:53.841332,2
2,10003,PREGO,BT10,100,50,z0019_2.csv,2026-10-10 18:33:25.037856,1
3,10003,PREGO,BT10,100,60,z0019_1.csv,2026-10-10 18:25:53.841332,2
4,10001,PARAFUSO,BT10,100,100,z0019_2.csv,2026-10-10 18:33:25.037856,1
5,10001,SERRA,BT50,100,200,z0019_1.csv,2026-10-10 18:25:53.841332,2


In [17]:
df_final = df.drop(columns=['nome_arquivo', 'data_injestao', 'row'])

df_final.head(10)

,NATBR,MAKTX,WERKS,MAINS,LABST
0,10002,MARTELO,BT50,100,1500
1,10002,MACHADO,BT50,100,100
2,10003,PREGO,BT10,100,50
3,10003,PREGO,BT10,100,60
4,10001,PARAFUSO,BT10,100,100
5,10001,SERRA,BT50,100,200


In [29]:
df_final = df_final.rename(columns={"NATBR": "id"})
df_final = df_final.rename(columns={"MAKTX": "nm produto"})
df_final = df_final.rename(columns={"WERKS": "id_categoria"})
df_final = df_final.rename(columns={"MAINS": "id_fornecedor"})
df_final = df_final.rename(columns={"LABST": "vl_preco"})
df_final = df_final.rename(columns={"nm produto": "nm_produto"})


df_final.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10002,MARTELO,BT50,100,1500
1,10002,MACHADO,BT50,100,100
2,10003,PREGO,BT10,100,50
3,10003,PREGO,BT10,100,60
4,10001,PARAFUSO,BT10,100,100
5,10001,SERRA,BT50,100,200


In [30]:
df2 = df_final
df_final.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10002,MARTELO,BT50,100,1500
1,10002,MACHADO,BT50,100,100
2,10003,PREGO,BT10,100,50
3,10003,PREGO,BT10,100,60
4,10001,PARAFUSO,BT10,100,100
5,10001,SERRA,BT50,100,200


In [31]:
df_final.dtypes

id               object
nm_produto       object
id_categoria     object
id_fornecedor    object
vl_preco         object
dtype: object

In [39]:
df2 = df_final
df2 = df2.astype(
    {
        'id' : int,
        'nm_produto': str,
        'id_categoria': str,
        'vl_preco': float,
        'id_fornecedor' : int
    }
)
df2.dtypes
#df2.head(10)

id                 int64
nm_produto        object
id_categoria      object
id_fornecedor      int64
vl_preco         float64
dtype: object

In [40]:
con.execute("""
CREATE TABLE IF NOT EXISTS produtos (
            id BIGINT,
            nm_produto TEXT,
            id_categoria TEXT,
            id_fornecedor BIGINT,
            vl_preco FLOAT
            )
""")

In [41]:
df2.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10002,MARTELO,BT50,100,1500.0
1,10002,MACHADO,BT50,100,100.0
2,10003,PREGO,BT10,100,50.0
3,10003,PREGO,BT10,100,60.0
4,10001,PARAFUSO,BT10,100,100.0
5,10001,SERRA,BT50,100,200.0


In [45]:
con.execute("INSERT INTO produtos SELECT * from df2")

In [46]:
df_resultado = con.execute("select * from produtos").fetchdf()
df_resultado.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10002,MARTELO,BT50,100,1500.0
1,10002,MACHADO,BT50,100,100.0
2,10003,PREGO,BT10,100,50.0
3,10003,PREGO,BT10,100,60.0
4,10001,PARAFUSO,BT10,100,100.0
5,10001,SERRA,BT50,100,200.0
6,10002,MARTELO,BT50,100,1500.0
7,10002,MACHADO,BT50,100,100.0
8,10003,PREGO,BT10,100,50.0
9,10003,PREGO,BT10,100,60.0


In [47]:
con.close()